# Gradio Demo: Compare ViT5 Stage 2 vs BARTpho Stage 2

Notebook nay dung de so sanh output tom tat giua hai model Stage 2:

- **ViT5 Stage 2**: thuong nam o `NLP_Final/Output stage 2 vit5 checkpoint/best_model` hoac `ViT5_stage_2/runs/vit5_base_stage2/checkpoints/checkpoint-735`.
- **BARTpho Stage 2**: thuong nam o `NLP_Final/BARTpho_stage_2/runs/bartpho_word_stage2/best_model`.

Notebook co co che tu do path tren Colab Drive va local workspace. Neu folder model thieu file weight (`model.safetensors` hoac `pytorch_model.bin`), notebook se bao ro path dang thieu.

## 1. Install dependencies if needed

Chi dat `INSTALL_DEPS = True` khi runtime moi thieu package. Sau khi install tren Colab, restart runtime roi chay tiep.

In [ ]:
INSTALL_DEPS = True

if INSTALL_DEPS:
    import sys, subprocess
    packages = [
        "gradio==4.44.1",
        "transformers==4.46.3",
        "tokenizers==0.20.3",
        "sentencepiece==0.2.0",
        "safetensors==0.4.5",
        "accelerate==1.1.1",
        "protobuf",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *packages])
    # Text summarization does not need torchvision. Removing it avoids CUDA
    # mismatch errors when pip/Colab provide different torch and torchvision builds.
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchvision"], check=False)
    print("Install done. Restart runtime roi chay lai notebook tu dau neu vua thay doi package.")


## 2. Mount Drive and detect paths

Neu notebook do sai path, sua truc tiep `VIT5_MODEL_DIR` va `BARTPHO_MODEL_DIR` o cuoi cell nay.

In [ ]:
from pathlib import Path
import os

MOUNT_DRIVE = True

try:
    from google.colab import drive
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB and MOUNT_DRIVE:
    drive.mount("/content/drive")

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "notebooks").exists() and (PROJECT_ROOT.parent / "notebooks").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DRIVE_ROOT = Path("/content/drive/MyDrive/NLP_Final")
LEGACY_ROOT = Path("/content/drive/MyDrive/C.Science - TDTU/NLP/Finalterm NLP")

WEIGHT_FILES = ["model.safetensors", "pytorch_model.bin", "tf_model.h5", "flax_model.msgpack"]

def find_model_dirs(root):
    root = Path(root)
    if not root.exists():
        return []
    found = []
    for cfg in root.rglob("config.json"):
        model_dir = cfg.parent
        if any((model_dir / name).exists() for name in WEIGHT_FILES):
            found.append(model_dir)
    return sorted(found, key=lambda p: ("best_model" not in str(p), len(str(p))))

def has_config(path):
    path = Path(path)
    return (path / "config.json").exists()

def has_weights(path):
    path = Path(path)
    return any((path / name).exists() for name in WEIGHT_FILES)

def pick_model_dir(candidates):
    candidates = [Path(p) for p in candidates]
    for path in candidates:
        if has_config(path) and has_weights(path):
            return path
    for path in candidates:
        if has_config(path):
            return path
    return candidates[0]

vit5_candidates = [
    DRIVE_ROOT / "Output stage 2 vit5 checkpoint" / "best_model",
    DRIVE_ROOT / "Output stage 2 vit5 checkpoint" / "model_for_rl",
    DRIVE_ROOT / "ViT5_stage_2" / "runs" / "vit5_base_stage2" / "best_model",
    DRIVE_ROOT / "ViT5_stage_2" / "runs" / "vit5_base_stage2" / "checkpoints" / "checkpoint-735",
    LEGACY_ROOT / "ViT5_stage_2" / "runs" / "vit5_base_stage2" / "checkpoints" / "checkpoint-735",
    PROJECT_ROOT / "ViT5_stage_2" / "runs" / "vit5_base_stage2" / "checkpoints" / "checkpoint-735",
    PROJECT_ROOT / "best_model",
]

bartpho_candidates = [
    *find_model_dirs(DRIVE_ROOT / "BARTpho_stage_2"),
    *find_model_dirs(LEGACY_ROOT / "BARTpho_stage_2"),
    *find_model_dirs(PROJECT_ROOT / "BARTpho_stage_2"),
    DRIVE_ROOT / "BARTpho_stage_2" / "runs" / "bartpho_word_stage2" / "best_model",
    DRIVE_ROOT / "Output stage 2 bartpho word" / "runs" / "bartpho_word_stage2" / "best_model",
    DRIVE_ROOT / "Output stage 2 bartpho checkpoint" / "runs" / "bartpho_word_stage2" / "best_model",
    LEGACY_ROOT / "BARTpho_stage_2" / "runs" / "bartpho_word_stage2" / "best_model",
    PROJECT_ROOT / "BARTpho_stage_2" / "runs" / "bartpho_word_stage2" / "best_model",
]

VIT5_MODEL_DIR = pick_model_dir(vit5_candidates)
BARTPHO_MODEL_DIR = pick_model_dir(bartpho_candidates)

data_candidates = [
    Path("/content/drive/MyDrive/data/test/dataset.jsonl"),
    DRIVE_ROOT / "data" / "test" / "dataset.jsonl",
    DRIVE_ROOT / "BARTpho_stage_2" / "data" / "test" / "dataset.jsonl",
    PROJECT_ROOT / "data" / "test" / "dataset.jsonl",
    PROJECT_ROOT / "BARTpho_stage_2" / "data" / "test" / "dataset.jsonl",
    PROJECT_ROOT / "data" / "val" / "dataset.jsonl",
]
DATA_PATH = next((p for p in data_candidates if p.exists()), data_candidates[0])

bartpho_pred_candidates = [
    DRIVE_ROOT / "BARTpho_stage_2" / "runs" / "bartpho_word_stage2" / "results" / "stage2_predictions.jsonl",
    DRIVE_ROOT / "BARTpho_stage_2" / "runs" / "bartpho_word_stage2" / "results" / "final_results_stage2.json",
    LEGACY_ROOT / "BARTpho_stage_2" / "runs" / "bartpho_word_stage2" / "results" / "stage2_predictions.jsonl",
    PROJECT_ROOT / "BARTpho_stage_2" / "runs" / "bartpho_word_stage2" / "results" / "stage2_predictions.jsonl",
    PROJECT_ROOT / "BARTpho_stage_2" / "runs" / "bartpho_word_stage2" / "results" / "final_results_stage2.json",
]
BARTPHO_PRED_PATH = next((p for p in bartpho_pred_candidates if p.exists()), bartpho_pred_candidates[0])

def describe_model(label, path):
    status = "OK" if has_config(path) and has_weights(path) else "MISSING WEIGHTS" if has_config(path) else "NOT FOUND"
    print(f"{label}: {path}")
    print(f"  status={status}, config={has_config(path)}, weights={has_weights(path)}")

describe_model("ViT5 Stage 2", VIT5_MODEL_DIR)
describe_model("BARTpho Stage 2", BARTPHO_MODEL_DIR)
print("Data path:", DATA_PATH if DATA_PATH.exists() else "not found, manual input still works")
print("BARTpho cached predictions:", BARTPHO_PRED_PATH if BARTPHO_PRED_PATH.exists() else "not found")


## 3. Imports, helpers, and sample data

In [ ]:
import os
os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"
os.environ["TRANSFORMERS_NO_VISION"] = "1"

import gc
import json
import random
import re
import time
from pathlib import Path

import gradio as gr
import torch
from transformers import AutoModelForSeq2SeqLM, PhobertTokenizer, T5Tokenizer

MAX_INPUT_LEN = 1024
NO_REPEAT_NGRAM = 3
TARGET_RATIO = 0.225
NUM_RE = re.compile(r"\d+(?:[.,]\d+)*")

def normalize_text(text):
    return " ".join((text or "").split())

def extract_numbers(text):
    return set(NUM_RE.findall(text or ""))

def length_ratio(source, pred):
    return len(normalize_text(pred).split()) / max(len(normalize_text(source).split()), 1)

def length_score(source, pred, target=TARGET_RATIO):
    ratio = length_ratio(source, pred)
    if ratio <= target:
        return max(0.0, 1.0 - (target - ratio) / target)
    return max(0.0, 1.0 - 1.5 * (ratio - target) / target)

def number_recall(source, pred):
    src = extract_numbers(source)
    if not src:
        return 1.0
    return len(src & extract_numbers(pred)) / len(src)

def number_hallucination(source, pred):
    src = extract_numbers(source)
    out = extract_numbers(pred)
    if not out:
        return 0.0
    return len(out - src) / len(out)

def quality_score(source, pred):
    return 0.55 * number_recall(source, pred) + 0.35 * length_score(source, pred) - 0.55 * number_hallucination(source, pred)

def metric_row(name, source, summary, elapsed):
    src_nums = extract_numbers(source)
    out_nums = extract_numbers(summary)
    return [
        name,
        len(normalize_text(summary).split()),
        round(length_ratio(source, summary), 4),
        round(number_recall(source, summary), 4),
        round(number_hallucination(source, summary), 4),
        f"{len(src_nums & out_nums)}/{len(src_nums)}",
        len(out_nums - src_nums),
        round(quality_score(source, summary), 4),
        round(elapsed, 2),
    ]

def numbers_comparison(source, vit5_summary, bartpho_summary):
    src = extract_numbers(source)
    vit5 = extract_numbers(vit5_summary)
    bart = extract_numbers(bartpho_summary)
    rows = []
    for num in sorted(src | vit5 | bart)[:120]:
        rows.append([
            num,
            "yes" if num in src else "no",
            "yes" if num in vit5 else "no",
            "yes" if num in bart else "no",
        ])
    return rows if rows else [["-", "no numbers", "-", "-"]]

def pick_field(row, names):
    for name in names:
        value = row.get(name)
        if isinstance(value, str) and value.strip():
            return value.strip()
    return ""

def preview_label(source, fallback, max_chars=86):
    text = normalize_text(source)
    if not text:
        text = str(fallback or "Sample")
    text = text.replace("\n", " ").strip()
    return text[:max_chars] + ("..." if len(text) > max_chars else "")

def load_bartpho_prediction_records(path=BARTPHO_PRED_PATH):
    path = Path(path)
    if not path.exists():
        return []
    if path.suffix == ".jsonl":
        rows = []
        with path.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    rows.append(json.loads(line))
        return rows
    obj = json.loads(path.read_text(encoding="utf-8"))
    return obj.get("records", []) if isinstance(obj, dict) else []

BARTPHO_PRED_RECORDS = load_bartpho_prediction_records()
BARTPHO_CACHE_BY_SOURCE = {
    normalize_text(r.get("source", "")): r
    for r in BARTPHO_PRED_RECORDS
    if r.get("source")
}

def cached_bartpho_summary(source):
    row = BARTPHO_CACHE_BY_SOURCE.get(normalize_text(source))
    if not row:
        return None
    return normalize_text(row.get("pred_rerank") or row.get("pred_sft") or row.get("prediction") or "")

def load_examples(path=DATA_PATH, limit=12):
    if BARTPHO_PRED_RECORDS:
        rows = list(BARTPHO_PRED_RECORDS)
        random.Random(42).shuffle(rows)
        choices, mapping = [], {}
        for i, row in enumerate(rows[:limit], 1):
            source = pick_field(row, ["source", "text", "article", "document", "content", "src", "input"])
            if not source:
                continue
            title = pick_field(row, ["title"])
            reference = pick_field(row, ["summary", "target", "abstract", "tgt", "gold", "label"])
            label = f"{i}. {preview_label(source, title or row.get('id'))}"
            choices.append(label)
            mapping[label] = {"source": source, "reference": reference}
        return choices, mapping

    path = Path(path)
    if not path.exists():
        return [], {}
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    random.Random(42).shuffle(rows)
    choices, mapping = [], {}
    for i, row in enumerate(rows[:limit], 1):
        source = pick_field(row, ["source", "text", "article", "document", "content", "src", "input"])
        if not source:
            continue
        title = pick_field(row, ["title"])
        reference = pick_field(row, ["summary", "target", "abstract", "tgt", "gold", "label"])
        label = f"{i}. {preview_label(source, title or row.get('id'))}"
        choices.append(label)
        mapping[label] = {"source": source, "reference": reference}
    return choices, mapping

EXAMPLE_CHOICES, EXAMPLE_MAP = load_examples()
EXAMPLE_CHOICES[:3]


## 4. Model loading and generation

Hai model duoc load tuan tu de giam rui ro OOM GPU. Ham tokenizer co fallback tao ban tokenizer tam neu `added_tokens_decoder` gay `KeyError: 0` tren Colab.

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
print("Device:", DEVICE, "| dtype:", DTYPE)

def load_tokenizer(model_dir):
    model_dir = Path(model_dir)
    config = {}
    cfg_path = model_dir / "tokenizer_config.json"
    if cfg_path.exists():
        config = json.loads(cfg_path.read_text(encoding="utf-8"))

    # Avoid AutoTokenizer here. Some saved checkpoints contain
    # added_tokens_decoder with string keys, which triggers KeyError: 0
    # in several Colab/Transformers combinations.
    if (model_dir / "spiece.model").exists():
        return T5Tokenizer(
            vocab_file=str(model_dir / "spiece.model"),
            eos_token="</s>",
            unk_token="<unk>",
            pad_token="<pad>",
            extra_ids=int(config.get("extra_ids", 96)),
            model_max_length=MAX_INPUT_LEN,
            legacy=True,
        )

    if (model_dir / "vocab.txt").exists() and (model_dir / "bpe.codes").exists():
        return PhobertTokenizer(
            vocab_file=str(model_dir / "vocab.txt"),
            merges_file=str(model_dir / "bpe.codes"),
            bos_token="<s>",
            eos_token="</s>",
            sep_token="</s>",
            cls_token="<s>",
            unk_token="<unk>",
            pad_token="<pad>",
            mask_token="<mask>",
            model_max_length=MAX_INPUT_LEN,
        )

    raise FileNotFoundError(
        f"Cannot build tokenizer manually from {model_dir}. Need spiece.model for ViT5 or vocab.txt + bpe.codes for BARTpho."
    )

def validate_model_dir(label, model_dir):
    model_dir = Path(model_dir)
    if not (model_dir / "config.json").exists():
        raise FileNotFoundError(f"{label} missing config.json: {model_dir}")
    if not any((model_dir / name).exists() for name in WEIGHT_FILES):
        raise FileNotFoundError(
            f"{label} missing model weights in {model_dir}. Need one of: {', '.join(WEIGHT_FILES)}"
        )

def load_model_pair(label, model_dir):
    validate_model_dir(label, model_dir)
    tokenizer = load_tokenizer(model_dir)
    model = AutoModelForSeq2SeqLM.from_pretrained(str(model_dir), torch_dtype=DTYPE)
    model.to(DEVICE).eval()
    model.config.use_cache = True
    return tokenizer, model

def token_id(value, fallback):
    if isinstance(value, (list, tuple)):
        return value[0] if value else fallback
    return fallback if value is None else value

@torch.no_grad()
def generate_one(label, model_dir, source, beams=4, max_len=180, length_penalty=1.0):
    tokenizer, model = load_model_pair(label, Path(model_dir))
    batch = tokenizer(
        [source],
        max_length=MAX_INPUT_LEN,
        truncation=True,
        padding=True,
        return_tensors="pt",
    ).to(DEVICE)
    pad_id = token_id(tokenizer.pad_token_id, token_id(model.config.pad_token_id, 0))
    eos_id = token_id(tokenizer.eos_token_id, token_id(model.config.eos_token_id, 1))
    decoder_start_id = token_id(model.config.decoder_start_token_id, pad_id)
    ids = model.generate(
        **batch,
        max_length=int(max_len),
        num_beams=max(1, int(beams)),
        no_repeat_ngram_size=NO_REPEAT_NGRAM,
        length_penalty=float(length_penalty),
        early_stopping=int(beams) > 1,
        decoder_start_token_id=decoder_start_id,
        pad_token_id=pad_id,
        eos_token_id=eos_id,
        use_cache=True,
    )
    summary = normalize_text(tokenizer.batch_decode(ids, skip_special_tokens=True)[0])
    if not summary:
        raw = tokenizer.batch_decode(ids, skip_special_tokens=False)[0]
        summary = f"[EMPTY GENERATION] token_ids={ids[0][:40].tolist()} raw={raw!r}"
    del batch, ids, model, tokenizer
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    gc.collect()
    return summary

def compare_models(source, beams, max_len, length_penalty):
    try:
        source = normalize_text(source)
        if len(source.split()) < 30:
            metrics = [["Input too short", 0, 0, 0, 0, "0/0", 0, 0, 0]]
            return "", "", metrics, []

        start = time.time()
        vit5_summary = generate_one("ViT5 Stage 2", VIT5_MODEL_DIR, source, beams, max_len, length_penalty)
        vit5_elapsed = time.time() - start

        metrics = [metric_row("ViT5 Stage 2", source, vit5_summary, vit5_elapsed)]

        if has_weights(BARTPHO_MODEL_DIR):
            start = time.time()
            bartpho_summary = generate_one("BARTpho Stage 2", BARTPHO_MODEL_DIR, source, beams, max_len, length_penalty)
            bart_elapsed = time.time() - start
            metrics.append(metric_row("BARTpho Stage 2", source, bartpho_summary, bart_elapsed))
        else:
            bartpho_summary = cached_bartpho_summary(source)
            if bartpho_summary:
                metrics.append(metric_row("BARTpho Stage 2 cached", source, bartpho_summary, 0.0))
            else:
                bartpho_summary = (
                    "BARTpho Stage 2 dang dung che do cached vi thieu file weight.\n\n"
                    "Bai nay khong nam trong file predictions da luu, nen khong co output BARTpho.\n"
                    "Hay chon Bai mau tu dropdown de co output BARTpho cached."
                )
                metrics.append(["BARTpho Stage 2 cached", 0, 0, 0, 0, "no cached output", 0, 0, 0])

        numbers = numbers_comparison(source, vit5_summary, bartpho_summary)
        return vit5_summary, bartpho_summary, metrics, numbers
    except Exception as e:
        import traceback
        tb = traceback.format_exc()
        msg = f"Loi khi chay demo:\n{type(e).__name__}: {e}\n\n{tb[-4000:]}"
        return msg, msg, [], []

def load_example(choice):
    item = EXAMPLE_MAP.get(choice, {})
    return item.get("source", ""), item.get("reference", "")

def clear_all():
    return "", "", "", "", [], []


## 5. Gradio UI

In [ ]:
CSS = """
.gradio-container { max-width: 1450px !important; }
#model-paths { font-size: 12px; color: #475569; line-height: 1.6; }
textarea { font-size: 14px !important; line-height: 1.55 !important; }
"""

with gr.Blocks(theme=gr.themes.Soft(primary_hue="teal", secondary_hue="blue"), css=CSS, title="Compare Stage 2 Models") as demo:
    gr.Markdown("# So sanh tom tat: ViT5 Stage 2 vs BARTpho Stage 2")
    gr.HTML(f"""
    <div id='model-paths'>
      <b>ViT5 Stage 2:</b> {VIT5_MODEL_DIR}<br>
      <b>BARTpho Stage 2:</b> {BARTPHO_MODEL_DIR}<br>
      <b>Runtime:</b> {DEVICE}<br>
      <b>BARTpho predictions:</b> {BARTPHO_PRED_PATH if BARTPHO_PRED_PATH.exists() else 'not found'}<br>
      <b>Note:</b> Neu BARTpho thieu weight, demo se dung output cached tu file predictions cho cac bai mau.
    </div>
    """)

    with gr.Row():
        with gr.Column(scale=5):
            example = gr.Dropdown(
                choices=EXAMPLE_CHOICES,
                value=EXAMPLE_CHOICES[0] if EXAMPLE_CHOICES else None,
                label="Bai mau",
                interactive=True,
            )
            source = gr.Textbox(label="Noi dung bai bao", lines=18, placeholder="Dan noi dung bai bao tieng Viet vao day...")
            reference = gr.Textbox(label="Tom tat tham chieu (neu co trong dataset)", lines=5, interactive=False)
            with gr.Row():
                load_btn = gr.Button("Nap bai mau")
                run_btn = gr.Button("So sanh output", variant="primary")
                clear_btn = gr.Button("Xoa")
            with gr.Accordion("Cai dat generate", open=False):
                beams = gr.Slider(1, 8, value=4, step=1, label="Beam search")
                max_len = gr.Slider(64, 384, value=180, step=16, label="Max output length")
                length_penalty = gr.Slider(0.6, 1.6, value=1.0, step=0.1, label="Length penalty")

        with gr.Column(scale=7):
            with gr.Row():
                vit5_output = gr.Textbox(label="Output ViT5 Stage 2", lines=10, show_copy_button=True)
                bartpho_output = gr.Textbox(label="Output BARTpho Stage 2", lines=10, show_copy_button=True)
            metrics = gr.Dataframe(
                headers=["model", "words", "length_ratio", "number_recall", "number_hallucination", "kept_numbers", "new_numbers", "score", "time_sec"],
                datatype=["str", "number", "number", "number", "number", "str", "number", "number", "number"],
                label="Bang so sanh nhanh",
                wrap=True,
            )
            numbers = gr.Dataframe(
                headers=["number", "in_source", "vit5_stage2", "bartpho_stage2"],
                datatype=["str", "str", "str", "str"],
                label="Doi chieu so lieu trong source va output",
                wrap=True,
            )

    load_btn.click(load_example, inputs=example, outputs=[source, reference])
    example.change(load_example, inputs=example, outputs=[source, reference])
    run_btn.click(compare_models, inputs=[source, beams, max_len, length_penalty], outputs=[vit5_output, bartpho_output, metrics, numbers])
    clear_btn.click(clear_all, outputs=[source, reference, vit5_output, bartpho_output, metrics, numbers])

demo.queue(max_size=4).launch(share=IN_COLAB, show_error=True)
